# 数字1：krのモード密度を比較

kzの間隔を元・1/2・1/4に変更し、krの最小/最大と中心半径、ランク100、Thread配置、伝搬範囲0〜50 cmを固定します。標準条件のモード数は15・29・57です。各条件でFrozen Wave係数を再計算し、直接法・提案法のCGH再生を比較します。

**波数の帯域を広げる比較ではありません。** モード密度を増やしても縦方向の解像度が必ず改善するとは限りません。直接法と目標との差、提案法と直接法との差、シフト射影の誤差を別々に見ます。詳細は `LSH数字1_kr間隔比較.md`。

修正後は移動による奇成分も含めたshift_aware基底で比較し、旧center_onlyのv1結果は読み込みません。新しいshiftaware_v2の結果へ保存します。


In [ ]:
import importlib
import json
from dataclasses import asdict
from pathlib import Path
import lsh_digit1_theory as theory
import lsh_digit1_proposed as proposed
theory = importlib.reload(theory)
proposed = importlib.reload(proposed)

RANK = 100
SEED = 42
BASIS_STRATEGY = 'shift_aware'
SPACING_FACTORS = (1, 2, 4)
OUTPUT_VERSION = 'v2'  # 同じ条件の結果を読み込む。再計算はv2等へ変更。
reference_results_dir = Path('lsh_digit1_theory_outputs_v1')
config = (proposed.config_from_theory_results(reference_results_dir)
          if (reference_results_dir / 'summary.json').is_file() else theory.Digit1Config())
aggregate_dir = Path(f'lsh_digit1_kr_spacing_outputs_rank{RANK}_{OUTPUT_VERSION}')
print('実際の撮影範囲[mm]:', 0, config.design_length_m*1000)
for factor in SPACING_FACTORS:
    design = proposed.mode_design_for_spacing(config, factor)
    print('密度倍率:', factor, 'モード数:', design.count,
          'kr端点[1/m]:', design.kr_per_m[[0, -1]],
          'kz間隔[1/m]:', 2*theory.np.pi/(config.design_length_m*factor))


In [ ]:
expected_config = theory.config_payload(config)
summaries = {}
case_dirs = {}
for factor in SPACING_FACTORS:
    suffix = '' if factor == 1 else f'_spacing{factor}'
    suffix += '_shiftaware'
    output_dir = Path(f'lsh_digit1_proposed_rank{RANK}{suffix}_{OUTPUT_VERSION}')
    library_path = Path(f'lsh_digit1_library_rank{RANK}{suffix}.npz')
    summary_file = output_dir / 'summary.json'
    if summary_file.is_file() and (output_dir / 'manifest.json').is_file():
        summary = json.loads(summary_file.read_text(encoding='utf-8'))
        if (summary.get('study') != 'digit1_direct_vs_projected_shared_core'
                or summary.get('schema_version') != proposed.SCHEMA_VERSION
                or summary.get('rank') != RANK or summary.get('seed') != SEED
                or theory.config_payload(summary.get('config', {})) != expected_config
                or summary.get('mode_spacing_factor', 1) != factor
                or summary.get('basis_strategy', 'center_only') != BASIS_STRATEGY
                or summary.get('library_path') != str(library_path.resolve())
                or summary.get('shift_diagnostics', {}).get('status') == 'not_run'):
            raise ValueError(f'{output_dir}: 別条件または不完全な結果です。新しい保存先にしてください。')
        print('既存結果を読み込み:', output_dir)
    elif output_dir.exists() and any(output_dir.iterdir()):
        raise ValueError(f'{output_dir}: 途中結果があります。OUTPUT_VERSIONを変更してください。')
    else:
        summary = proposed.run_comparison(output_dir, config, rank=RANK,
            library_path=library_path, seed=SEED, mode_spacing_factor=factor, basis_strategy=BASIS_STRATEGY,
            run_shift_diagnostics=True)
    summaries[factor] = summary
    case_dirs[factor] = output_dir
    print('密度倍率:', factor, '再生後の提案/直接:', summary['metrics']['cgh_sheet_proposed_vs_direct'])


In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
rows = []
for factor, summary in summaries.items():
    metrics = summary['metrics']
    direct = metrics['direct_cgh_sheet_vs_target']
    rec = metrics['proposed_cgh_sheet_vs_target']
    pair = metrics['cgh_sheet_proposed_vs_direct']
    rows.append({
        'spacing_factor': factor, 'modes_per_thread': summary['modes_per_thread'],
        'kz_spacing_per_m': 2*np.pi/(config.design_length_m*factor),
        'direct_vs_target_intensity_l2_percent': 100*direct['intensity_relative_l2'],
        'proposed_vs_target_intensity_l2_percent': 100*rec['intensity_relative_l2'],
        'direct_vs_target_ssim': direct['ssim'], 'proposed_vs_target_ssim': rec['ssim'],
        'proposed_vs_direct_intensity_l2_percent': 100*pair['intensity_relative_l2'],
        'proposed_vs_direct_ssim': pair['ssim'], 'proposed_vs_direct_psnr_db': pair['psnr_db'],
        'shift_projection_complex_l2_percent': 100*summary['shift_diagnostics']['compressed_periodic_vs_projected_proposed']['after_aperture_complex_relative_l2'],
        'dictionary_compression_complex_l2_percent': 100*summary['shift_diagnostics']['exact_periodic_vs_compressed_periodic']['after_aperture_complex_relative_l2'],
        'direct_field_seconds': summary['timings_seconds']['direct_field_generation'],
        'proposed_field_seconds': summary['timings_seconds']['proposed_field_generation']['field_generation_seconds'],
    })
aggregate_dir.mkdir(parents=True, exist_ok=True)
with (aggregate_dir / 'spacing_comparison.csv').open('w', newline='', encoding='utf-8-sig') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
theory._write_json(aggregate_dir / 'spacing_comparison.json', {
    'study': 'digit1_same_kr_band_finer_kz_spacing', 'config': asdict(config),
    'rank': RANK, 'seed': SEED, 'results': rows,
    'basis_strategy': BASIS_STRATEGY,
    'case_directories': {str(f): str(p.resolve()) for f, p in case_dirs.items()},
    'note': 'Same kr endpoints and propagation range; coefficient period and quadrature count grow with spacing factor.'})
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
modes = [r['modes_per_thread'] for r in rows]
for key, label in (('direct_vs_target_intensity_l2_percent', 'Direct vs target'),
                   ('proposed_vs_target_intensity_l2_percent', 'Proposed vs target')):
    axes[0].plot(modes, [r[key] for r in rows], 'o-', label=label)
axes[0].set(title='Target reproduction', ylabel='Intensity relative L2 [%]')
axes[1].plot(modes, [r['proposed_vs_direct_intensity_l2_percent'] for r in rows], 'o-', label='Proposed vs direct intensity')
axes[1].plot(modes, [r['shift_projection_complex_l2_percent'] for r in rows], 'o-', label='Shift projection complex field')
axes[1].set(title='Proposed method error', ylabel='Relative L2 [%]')
axes[2].plot(modes, [r['direct_vs_target_ssim'] for r in rows], 'o-', label='Direct vs target')
axes[2].plot(modes, [r['proposed_vs_target_ssim'] for r in rows], 'o-', label='Proposed vs target')
axes[2].plot(modes, [r['proposed_vs_direct_ssim'] for r in rows], 'o-', label='Proposed vs direct')
axes[2].set(title='Intensity SSIM', ylabel='SSIM', ylim=(0, 1))
for ax in axes:
    ax.set_xlabel('Bessel modes per thread')
    ax.set_xticks(modes)
    ax.grid(alpha=.3)
    ax.legend()
fig.savefig(aggregate_dir / 'spacing_comparison.png', dpi=160)
plt.close(fig)
print('集計保存先:', aggregate_dir.resolve())
for row in rows:
    print(row)


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(aggregate_dir / 'spacing_comparison.png'), width=1300))
for factor, directory in case_dirs.items():
    print('密度倍率:', factor, 'モード数:', summaries[factor]['modes_per_thread'])
    display(Image(filename=str(directory / 'target_direct_proposed.png'), width=1300))
